# Softmax + Categorical Cross-Entropy — Derivation & Implementation

**Goal:** derive every formula from scratch, implement it using `Value` scalars,
verify each gradient against PyTorch, and train a 3-class classifier on Iris.

**Road map:**
1. Why do we need softmax? — the multi-class problem
2. Softmax: formula, intuition, numerical stability
3. Categorical cross-entropy: formula, MLE derivation
4. The magic collapse: why `dL/dz_i = p_i - y_i`
5. Implementing softmax in `engine.py`
6. Implementing CCE in `losses.py`
7. Verifying every gradient vs PyTorch and finite differences
8. Training `MLPClassifier` on Iris


## 1. The multi-class problem

Binary classification: sigmoid squashes one logit into `[0, 1]`.  
Multi-class ($C$ classes): we need $C$ outputs that are **non-negative** and **sum to 1**.

Sigmoid applied to $C$ logits independently does NOT sum to 1.  
We need a function that normalises across all classes simultaneously — that is **softmax**.


## 2. Softmax

### Formula

$$
p_i = \text{softmax}(z)_i = \frac{e^{z_i}}{\sum_{j=1}^{C} e^{z_j}}
$$

- Input: logit vector $z \in \mathbb{R}^C$ (raw scores from the network, any magnitude).
- Output: probability vector $p \in (0, 1)^C$ with $\sum_i p_i = 1$.

### Why exp?

We need outputs > 0. $e^x > 0$ always. The denominator normalises.

### Numerical stability: subtract the max

$e^{z_i}$ can overflow for large $z_i$ (e.g. $e^{1000} = \infty$ in float32).

Fix: subtract $C = \max_j z_j$ before exponentiation.

$$
p_i = \frac{e^{z_i - C}}{\sum_j e^{z_j - C}}
$$

The $C$ cancels in numerator and denominator, so **the output is identical** — but
the largest exponent is now $e^0 = 1$, so no overflow.


In [ ]:
import math

def softmax_naive(z):
    exps = [math.exp(zi) for zi in z]
    s = sum(exps)
    return [e / s for e in exps]

def softmax_stable(z):
    c = max(z)
    exps = [math.exp(zi - c) for zi in z]
    s = sum(exps)
    return [e / s for e in exps]

# Check they give the same result on normal inputs
z = [2.0, 1.0, 0.1]
print("naive: ", [f"{p:.6f}" for p in softmax_naive(z)])
print("stable:", [f"{p:.6f}" for p in softmax_stable(z)])
print("sum:   ", sum(softmax_stable(z)))


In [ ]:
# Naive overflows, stable does not
import math
z_big = [1000.0, 999.0, 998.0]
try:
    print("naive:", softmax_naive(z_big))
except OverflowError as e:
    print(f"naive OVERFLOWS: {e}")
print("stable:", [f"{p:.6f}" for p in softmax_stable(z_big)])


## 3. Categorical Cross-Entropy (MLE derivation)

We have $N$ samples, each with a true class $k_n \in \{0, \ldots, C-1\}$.

**Model:** $P(Y = c \mid x) = p_c = \text{softmax}(f(x))_c$.

**Log-likelihood** (sum over samples):

$$
\ell(\theta) = \sum_{n=1}^N \log P(y_n \mid x_n) = \sum_{n=1}^N \log p_{k_n}
$$

**Negative log-likelihood** (= loss to minimise):

$$
L = -\frac{1}{N} \sum_{n=1}^N \log p_{k_n}
$$

Using the one-hot encoding $y_{n,i} = 1$ if $i = k_n$, else 0:

$$
\boxed{L = -\frac{1}{N} \sum_{n=1}^N \sum_{i=1}^C y_{n,i} \log p_{n,i}}
$$

This is **exactly** the KL divergence between the data distribution and the model
(plus a constant), so minimising CCE = maximising likelihood.


## 4. The magic collapse: $\frac{\partial L}{\partial z_i} = p_i - y_i$

This is the most beautiful fact in classification. Derive it carefully.

**Setup:** single sample, true class $k$, loss $L = -\log p_k$.

Step 1: write $p_k$ explicitly:

$$
p_k = \frac{e^{z_k}}{Z}, \quad Z = \sum_j e^{z_j}
$$

Step 2: take the log:

$$
L = -\log p_k = -(z_k - \log Z) = -z_k + \log Z
$$

Step 3: differentiate w.r.t. $z_i$ (for **any** $i$, not just $i = k$):

$$
\frac{\partial L}{\partial z_i} = \frac{\partial}{\partial z_i}(-z_k + \log Z)
= -\mathbb{1}[i = k] + \frac{1}{Z} \frac{\partial Z}{\partial z_i}
= -\mathbb{1}[i = k] + \frac{e^{z_i}}{Z}
= p_i - \mathbb{1}[i = k]
$$

Using one-hot $y_i = \mathbb{1}[i = k]$:

$$
\boxed{\frac{\partial L}{\partial z_i} = p_i - y_i}
$$

**Interpretation:**
- For the true class ($i = k$): gradient is $p_k - 1 < 0$. Push logit $z_k$ up.
- For wrong classes ($i \neq k$): gradient is $p_i > 0$. Push logit $z_i$ down.
- When $p_k \to 1$ (perfect prediction): all gradients $\to 0$. Training stops naturally.

For mean loss over $N$ samples, the gradient is $(p_i - y_i) / N$ (linearity of differentiation).


In [ ]:
# Verify the identity numerically on one example
import torch
import torch.nn.functional as F

z_data = [2.0, 1.0, 0.1]
k = 0  # true class

# Analytical: p_i - y_i
probs = softmax_stable(z_data)
y_onehot = [1.0 if i == k else 0.0 for i in range(3)]
analytical_grads = [p - y for p, y in zip(probs, y_onehot)]

# PyTorch autograd
t_z = torch.tensor(z_data, dtype=torch.float64, requires_grad=True)
t_loss = F.cross_entropy(t_z.unsqueeze(0), torch.tensor([k]))
t_loss.backward()
pt_grads = t_z.grad.tolist()

print("Class | p_i    | y_i | p_i - y_i | PyTorch grad")
print("------+--------+-----+-----------+-------------")
for i in range(3):
    print(f"  {i}   | {probs[i]:.4f} | {y_onehot[i]:.0f}   | {analytical_grads[i]:+.6f} | {pt_grads[i]:+.6f}")


## 5. Implementing softmax in `engine.py`

`Value.softmax` is a **static method** — it takes a list of Value logits and
returns a list of Value probabilities. Gradients flow automatically through
the existing `exp`, `*`, `**` primitives.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from microgradplus.engine import Value

# Forward pass
logits = [Value(2.0, label='z0'), Value(1.0, label='z1'), Value(0.1, label='z2')]
probs = Value.softmax(logits)

print("Probabilities:")
for i, p in enumerate(probs):
    print(f"  p_{i} = {p.data:.6f}")
print(f"  sum = {sum(p.data for p in probs):.10f}")


In [ ]:
# Backward pass — verify gradient is p_i - y_i
k = 0  # true class
loss = probs[k].log() * -1.0   # -log(p_k), single sample
loss.backward()

probs_vals = [p.data for p in probs]
y_onehot   = [1.0 if i == k else 0.0 for i in range(3)]

print("Gradient check (single sample, no mean normalisation):")
print("i | z_i.grad  | p_i - y_i | match?")
for i, l in enumerate(logits):
    expected = probs_vals[i] - y_onehot[i]
    ok = abs(l.grad - expected) < 1e-9
    print(f"{i} | {l.grad:+.7f} | {expected:+.7f} | {'✓' if ok else '✗'}")


## 6. Implementing `categorical_cross_entropy` in `losses.py`

```python
def categorical_cross_entropy(logits_batch, ytrue):
    n = len(logits_batch)
    total = Value(0.0)
    for logits, k in zip(logits_batch, ytrue):
        probs = Value.softmax(logits)
        total = total + probs[k].log()
    return total * (-1.0 / n)
```

Key design choices:
- **Softmax inside the loss** — the model outputs raw logits. Consistent with PyTorch's `CrossEntropyLoss`.
- **Mean** not sum — gradient magnitude stays stable across batch sizes.
- **Clamp** `p_k` away from 0 to avoid `log(0) = -inf`.


In [ ]:
from microgradplus.losses import categorical_cross_entropy

# Batch of 3 samples, 3 classes
batch_data = [[2.0, 1.0, 0.1], [0.5, 2.5, 0.3], [-1.0, 0.0, 3.0]]
targets    = [0, 1, 2]

logits_batch = [[Value(z) for z in row] for row in batch_data]
loss = categorical_cross_entropy(logits_batch, targets)
print(f"CCE loss = {loss.data:.6f}")

# Compare with PyTorch
import torch
t_logits = torch.tensor(batch_data, dtype=torch.float64)
t_targets = torch.tensor(targets, dtype=torch.long)
pt_loss = torch.nn.CrossEntropyLoss()(t_logits, t_targets)
print(f"PyTorch  = {pt_loss.item():.6f}")
print(f"Matches: {abs(loss.data - pt_loss.item()) < 1e-7}")


## 7. Full gradient verification

### 7a. Against PyTorch autograd


In [ ]:
# Verify every gradient in the batch against PyTorch
logits_batch2 = [[Value(z) for z in row] for row in batch_data]
loss2 = categorical_cross_entropy(logits_batch2, targets)
loss2.backward()

t_logits2 = torch.tensor(batch_data, dtype=torch.float64, requires_grad=True)
pt_loss2 = torch.nn.CrossEntropyLoss()(t_logits2, torch.tensor(targets, dtype=torch.long))
pt_loss2.backward()

print("Sample | Class | microgradplus | PyTorch    | Match?")
print("-------+-------+---------------+------------+-------")
for n in range(len(batch_data)):
    for i in range(3):
        mg = logits_batch2[n][i].grad
        pt = t_logits2.grad[n][i].item()
        ok = abs(mg - pt) < 1e-7
        print(f"   {n}   |   {i}   | {mg:+.9f} | {pt:+.9f} | {'✓' if ok else '✗ FAIL'}")


### 7b. Against finite differences

In [ ]:
def finite_diff(fn, z_list, idx, eps=1e-5):
    z_plus  = z_list[:idx] + [z_list[idx] + eps] + z_list[idx+1:]
    z_minus = z_list[:idx] + [z_list[idx] - eps] + z_list[idx+1:]
    return (fn(z_plus) - fn(z_minus)) / (2 * eps)

row = [2.0, 1.0, 0.1]
k   = 0

def loss_fn(z):
    logits = [Value(zi) for zi in z]
    loss   = categorical_cross_entropy([logits], [k])
    return loss.data

# Autograd
logits_fd = [Value(z) for z in row]
loss_fd   = categorical_cross_entropy([logits_fd], [k])
loss_fd.backward()

print("Class | autograd   | fin-diff   | error")
for i in range(3):
    fd  = finite_diff(loss_fn, row, i)
    ag  = logits_fd[i].grad
    err = abs(ag - fd)
    print(f"  {i}   | {ag:+.7f} | {fd:+.7f} | {err:.2e}")


## 8. Training MLPClassifier on Iris

Architecture: `[2 inputs] → [16 ReLU] → [16 ReLU] → [3 linear logits]`  
Loss: `categorical_cross_entropy`  
Optimizer: `Adam(lr=0.02)`


In [ ]:
import random
import numpy as np
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from microgradplus.nn import MLPClassifier
from microgradplus.losses import categorical_cross_entropy
from microgradplus.optim import Adam
from microgradplus.training import fit, predict_classes

random.seed(42)
np.random.seed(42)

iris = load_iris()
X_raw, y_raw = iris.data[:, :2], iris.target
scaler = StandardScaler()
X_sc = scaler.fit_transform(X_raw).tolist()
y    = y_raw.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X_sc, y, test_size=0.2, random_state=42, stratify=y
)

model = MLPClassifier(nin=2, hidden=[16, 16], n_classes=3,
                      activation="relu", init="he")
opt = Adam(model.parameters(), lr=0.02)

history = fit(model, opt, categorical_cross_entropy,
              X_train, y_train,
              epochs=300, batch_size=16, log_every=50,
              multiclass=True)


In [ ]:
y_pred = predict_classes(model, X_test)
acc = sum(p == t for p, t in zip(y_pred, y_test)) / len(y_test)
print(f"Test accuracy: {acc:.1%}")


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(history, color="#2563EB", lw=1.5)
axes[0].set(xlabel="Epoch", ylabel="CCE Loss", title="Training Loss")
axes[0].grid(True, alpha=0.3)

# Decision boundary
h = 0.05
x0 = np.array([x[0] for x in X_sc])
x1 = np.array([x[1] for x in X_sc])
xx, yy = np.meshgrid(
    np.arange(x0.min()-0.5, x0.max()+0.5, h),
    np.arange(x1.min()-0.5, x1.max()+0.5, h),
)
grid = [[float(a), float(b)] for a, b in zip(xx.ravel(), yy.ravel())]
Z = np.array(predict_classes(model, grid)).reshape(xx.shape)

colors_bg = ["#DBEAFE", "#D1FAE5", "#FEF3C7"]
colors_pt = ["#1D4ED8", "#047857", "#B45309"]
axes[1].contourf(xx, yy, Z, alpha=0.4,
                 colors=colors_bg, levels=[-0.5, 0.5, 1.5, 2.5])
for c, (bg, pt, name) in enumerate(zip(colors_bg, colors_pt, iris.target_names)):
    pts = [(xv[0], xv[1]) for xv, lv in zip(X_sc, y) if lv == c]
    axes[1].scatter([p[0] for p in pts], [p[1] for p in pts],
                    color=pt, label=name, s=25, edgecolors="white", lw=0.5)
axes[1].set(xlabel="Feature 1 (scaled)", ylabel="Feature 2 (scaled)",
            title=f"Decision boundary — test acc {acc:.0%}")
axes[1].legend(); axes[1].grid(True, alpha=0.2)
plt.tight_layout()
plt.show()


## Summary

| Step | Formula | Key insight |
|------|---------|-------------|
| Softmax | $p_i = e^{z_i} / \sum_j e^{z_j}$ | Normalises logits to probabilities |
| Stability | Subtract $\max(z)$ first | No overflow, identical output |
| CCE loss | $L = -(1/N)\sum_n \log p_{k_n}$ | MLE under categorical distribution |
| Gradient | $\partial L/\partial z_i = p_i - y_i$ | True class: push up. Others: push down. |
| Architecture | Linear output layer + CCE | Never put softmax inside the model |

The gradient $p_i - y_i$ is why softmax + CCE is the standard for multi-class classification:
it is simple, numerically stable, and has a clean probabilistic interpretation.
